# Problem 1: Automatic Dictionary Construction for NER

- Dataset: CoNLL-2003

- Develop a Python program to automatically construct three entity dictionaries (PERSON, LOCATION and ORGANIZATION) from the annotated training dataset. Use these dictionaries to identify entities in 200 unseen test sentences.

- Tasks: Extract annotated entity phrases, create unique dictionaries, apply longest-phrase matching and calculate entity-level precision, recall and F1-score.

- Expected output: Dictionary sizes, predicted entities, correctly identified entities and evaluation scores.

In [ ]:
!pip install -q datasets

# Dataset: CoNLL-2003

from datasets import load_dataset
from collections import defaultdict

In [ ]:
# --------------------------------------------------
# STEP 1: Load the dataset
# --------------------------------------------------

dataset = load_dataset("BramVanroy/conll2003")

train_data = dataset["train"]
test_data = dataset["test"]

# CoNLL-2003 label mapping:
# B-PER/I-PER = Person
# B-LOC/I-LOC = Location
# B-ORG/I-ORG = Organization

label_names = train_data.features["ner_tags"].feature.names

ENTITY_MAP = {
    "PER": "PERSON",
    "LOC": "LOCATION",
    "ORG": "ORGANIZATION"
}

print("Training sentences:", len(train_data))
print("Test sentences:", len(test_data))

Training sentences: 14041
Test sentences: 3453


In [ ]:
# --------------------------------------------------
# STEP 2: Extract entity phrases from BIO annotations
# --------------------------------------------------

def extract_entities(tokens, tags):
    """
    Extract entities as (start, end, type, phrase).
    start is inclusive; end is exclusive.
    """
    entities = []
    current_tokens = []
    current_type = None
    start = None

    for i, (token, tag_id) in enumerate(zip(tokens, tags)):
        tag = label_names[tag_id]

        if tag.startswith("B-"):
            # Close the previous entity
            if current_type is not None:
                entities.append((
                    start, i, current_type,
                    " ".join(current_tokens)
                ))

            entity_type = tag[2:]

            if entity_type in ENTITY_MAP:
                current_type = ENTITY_MAP[entity_type]
                current_tokens = [token]
                start = i
            else:
                current_type = None
                current_tokens = []
                start = None

        elif tag.startswith("I-"):
            entity_type = tag[2:]

            if (
                current_type is not None
                and ENTITY_MAP.get(entity_type) == current_type
            ):
                current_tokens.append(token)
            else:
                # Handle a malformed I-tag as a new entity
                if current_type is not None:
                    entities.append((
                        start, i, current_type,
                        " ".join(current_tokens)
                    ))

                if entity_type in ENTITY_MAP:
                    current_type = ENTITY_MAP[entity_type]
                    current_tokens = [token]
                    start = i
                else:
                    current_type = None
                    current_tokens = []
                    start = None

        else:  # O tag
            if current_type is not None:
                entities.append((
                    start, i, current_type,
                    " ".join(current_tokens)
                ))

            current_type = None
            current_tokens = []
            start = None

    # Close an entity at the end of the sentence
    if current_type is not None:
        entities.append((
            start, len(tokens), current_type,
            " ".join(current_tokens)
        ))

    return entities

In [ ]:
# --------------------------------------------------
# STEP 3: Construct three unique dictionaries
# --------------------------------------------------

entity_dict = {
    "PERSON": set(),
    "LOCATION": set(),
    "ORGANIZATION": set()
}

for example in train_data:
    tokens = example["tokens"]
    tags = example["ner_tags"]

    for start, end, entity_type, phrase in extract_entities(
        tokens, tags
    ):
        # Store normalized token sequences for matching
        normalized = tuple(
            word.casefold() for word in tokens[start:end]
        )
        entity_dict[entity_type].add(normalized)

print("\nDICTIONARY SIZES")
for entity_type, phrases in entity_dict.items():
    print(f"{entity_type}: {len(phrases)} unique phrases")


DICTIONARY SIZES
PERSON: 3531 unique phrases
LOCATION: 1106 unique phrases
ORGANIZATION: 2312 unique phrases


In [ ]:
# --------------------------------------------------
# STEP 4: Longest-phrase matching
# --------------------------------------------------

def predict_entities(tokens):
    """
    Match dictionary phrases against a sentence.
    At each position, choose the longest matching phrase.
    """
    normalized_tokens = [t.casefold() for t in tokens]
    predictions = []
    i = 0

    # Longer phrases are considered before shorter phrases.
    # The priority resolves equal-length cross-type matches.
    priority = ["PERSON", "LOCATION", "ORGANIZATION"]

    while i < len(tokens):
        matches = []

        for entity_type in priority:
            for phrase in entity_dict[entity_type]:
                length = len(phrase)

                if (
                    length > 0
                    and i + length <= len(tokens)
                    and tuple(normalized_tokens[i:i + length]) == phrase
                ):
                    matches.append((length, entity_type, phrase))

        if matches:
            # Longest match wins; priority breaks ties
            length, entity_type, phrase = sorted(
                matches,
                key=lambda x: (-x[0], priority.index(x[1]))
            )[0]

            predictions.append((
                i,
                i + length,
                entity_type,
                " ".join(tokens[i:i + length])
            ))

            # Skip matched tokens to prevent overlapping predictions
            i += length
        else:
            i += 1

    return predictions

In [ ]:
# --------------------------------------------------
# STEP 5: Evaluate on exactly 200 test sentences
# --------------------------------------------------

NUM_SENTENCES = 200

if len(test_data) < NUM_SENTENCES:
    raise ValueError("The test dataset contains fewer than 200 sentences.")

TP = 0
FP = 0
FN = 0

correct_entities = []
false_positives = []
false_negatives = []

for sentence_number in range(NUM_SENTENCES):
    example = test_data[sentence_number]
    tokens = example["tokens"]
    tags = example["ner_tags"]

    predicted = predict_entities(tokens)

    gold = extract_entities(tokens, tags)

    # Span + entity type must match exactly.
    predicted_set = {
        (start, end, entity_type)
        for start, end, entity_type, _ in predicted
    }

    gold_set = {
        (start, end, entity_type)
        for start, end, entity_type, _ in gold
    }

    true_positives = predicted_set & gold_set
    false_positive_set = predicted_set - gold_set
    false_negative_set = gold_set - predicted_set

    TP += len(true_positives)
    FP += len(false_positive_set)
    FN += len(false_negative_set)

    predicted_lookup = {
        (start, end, entity_type): phrase
        for start, end, entity_type, phrase in predicted
    }
    gold_lookup = {
        (start, end, entity_type): phrase
        for start, end, entity_type, phrase in gold
    }

    for key in sorted(true_positives):
        correct_entities.append((
            sentence_number + 1,
            key[2],
            gold_lookup[key]
        ))

    for key in sorted(false_positive_set):
        false_positives.append((
            sentence_number + 1,
            key[2],
            predicted_lookup[key]
        ))

    for key in sorted(false_negative_set):
        false_negatives.append((
            sentence_number + 1,
            key[2],
            gold_lookup[key]
        ))

    # Print predictions for every test sentence
    print(f"\nSentence {sentence_number + 1}:")
    print("Text:", " ".join(tokens))
    print("Predicted:", [
        (phrase, entity_type)
        for _, _, entity_type, phrase in predicted
    ])
    print("Gold:", [
        (phrase, entity_type)
        for _, _, entity_type, phrase in gold
    ])


Sentence 1:
Text: SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .
Predicted: [('JAPAN', 'LOCATION'), ('CHINA', 'LOCATION')]
Gold: [('JAPAN', 'LOCATION'), ('CHINA', 'PERSON')]

Sentence 2:
Text: Nadim Ladki
Predicted: []
Gold: [('Nadim Ladki', 'PERSON')]

Sentence 3:
Text: AL-AIN , United Arab Emirates 1996-12-06
Predicted: [('United Arab Emirates', 'LOCATION')]
Gold: [('AL-AIN', 'LOCATION'), ('United Arab Emirates', 'LOCATION')]

Sentence 4:
Text: Japan began the defence of their Asian Cup title with a lucky 2-1 win against Syria in a Group C championship match on Friday .
Predicted: [('Japan', 'LOCATION'), ('the', 'LOCATION'), ('defence', 'ORGANIZATION'), ('Syria', 'LOCATION')]
Gold: [('Japan', 'LOCATION'), ('Syria', 'LOCATION')]

Sentence 5:
Text: But China saw their luck desert them in the second match of the group , crashing to a surprise 2-0 defeat to newcomers Uzbekistan .
Predicted: [('China', 'LOCATION'), ('the', 'LOCATION'), ('the', 'LOCATION')]
Gold: [('China', 'LO

In [ ]:
# --------------------------------------------------
# STEP 6: Calculate precision, recall and F1-score
# --------------------------------------------------

precision = TP / (TP + FP) if TP + FP else 0.0
recall = TP / (TP + FN) if TP + FN else 0.0

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0.0
)

print("\n" + "=" * 50)
print("FINAL EVALUATION RESULTS")
print("=" * 50)

print("Test sentences evaluated:", NUM_SENTENCES)
print("Correctly identified entities (TP):", TP)
print("Incorrect predicted entities (FP):", FP)
print("Missed gold entities (FN):", FN)

print(f"Precision: {precision:.4f} ({precision * 100:.2f}%)")
print(f"Recall:    {recall:.4f} ({recall * 100:.2f}%)")
print(f"F1-score:  {f1:.4f} ({f1 * 100:.2f}%)")

print("\nCORRECTLY IDENTIFIED ENTITIES")
for sentence_no, entity_type, phrase in correct_entities:
    print(f"Sentence {sentence_no}: {phrase} [{entity_type}]")

print("\nFALSE POSITIVES")
for sentence_no, entity_type, phrase in false_positives:
    print(f"Sentence {sentence_no}: {phrase} [{entity_type}]")

print("\nFALSE NEGATIVES")
for sentence_no, entity_type, phrase in false_negatives:
    print(f"Sentence {sentence_no}: {phrase} [{entity_type}]")



FINAL EVALUATION RESULTS
Test sentences evaluated: 200
Correctly identified entities (TP): 160
Incorrect predicted entities (FP): 161
Missed gold entities (FN): 212
Precision: 0.4984 (49.84%)
Recall:    0.4301 (43.01%)
F1-score:  0.4618 (46.18%)

CORRECTLY IDENTIFIED ENTITIES
Sentence 1: JAPAN [LOCATION]
Sentence 3: United Arab Emirates [LOCATION]
Sentence 4: Japan [LOCATION]
Sentence 4: Syria [LOCATION]
Sentence 5: China [LOCATION]
Sentence 6: China [LOCATION]
Sentence 10: Japan [LOCATION]
Sentence 10: Syria [LOCATION]
Sentence 12: Syria [LOCATION]
Sentence 14: Syria [LOCATION]
Sentence 15: Japan [LOCATION]
Sentence 17: Japan [LOCATION]
Sentence 20: Japan [LOCATION]
Sentence 20: FIFA [ORGANIZATION]
Sentence 21: UAE [LOCATION]
Sentence 21: Kuwait [LOCATION]
Sentence 21: South Korea [LOCATION]
Sentence 21: Indonesia [LOCATION]
Sentence 23: RUGBY UNION [ORGANIZATION]
Sentence 23: ITALY [LOCATION]
Sentence 24: ROME [LOCATION]
Sentence 25: Italy [LOCATION]
Sentence 26: Scotland [LOCATION]